# Mitigation Overhead: Redundant PA and TMR

Measures the actual compute/time cost of the two mitigations, which nothing in the pipeline
currently records. Two measurements, kept separate on purpose:

- **Compute overhead** (local, via `MockClassicalSession`) -- isolates the real cost of
  doing the extra work, with no network noise. This is the number to cite in the paper.
- **Real-channel overhead** (FABRIC) -- wall-clock including actual network round-trips.
  Useful context (this is what a real deployment would feel), but noisy run-to-run and not
  a clean multiplier claim on its own.

Run `20_data_cleaning_and_integrity.ipynb` first for `key_pairs_metadata.csv`, or point
`RESULTS` at wherever your key metadata lives.

In [1]:
import sys, time
from pathlib import Path
import numpy as np
import pandas as pd

PROJECT_DIR = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(PROJECT_DIR))
sys.path.insert(0, str(PROJECT_DIR / 'scripts'))

from galois import GF2
from randextract import ToeplitzHashing
from qne.cascade import Key, ORIGINAL, Reconciliation, MockClassicalSession
from qne.cascade.key import key_from_sifted_json

RESULTS = PROJECT_DIR / "results"
key_pairs_df = pd.read_csv(str(RESULTS / "key_pairs_metadata.csv"))


## Redundant PA: compute overhead

In [2]:
N_TRIALS = 30
krow = key_pairs_df.iloc[0]
n_bits, ell = int(krow["n_bits"]), int(krow["n_bits"] * 0.9)  # adjust ell to your real output length

alice_bits = GF2(np.random.default_rng(0).integers(0, 2, n_bits))
ext = ToeplitzHashing(input_length=n_bits, output_length=ell)

single_times, redundant_times = [], []
for trial in range(N_TRIALS):
    seed_gf2 = GF2.Random(ext.seed_length)

    t0 = time.perf_counter()
    _ = ext.extract(alice_bits, seed_gf2)
    single_times.append(time.perf_counter() - t0)

    t0 = time.perf_counter()
    _ = ext.extract(alice_bits, seed_gf2)
    _ = ext.extract(alice_bits, seed_gf2)   # the second, redundant extraction
    redundant_times.append(time.perf_counter() - t0)

single_mean, redundant_mean = np.mean(single_times), np.mean(redundant_times)
print(f"Single PA extraction:    {single_mean*1000:.2f} ms  (n={N_TRIALS})")
print(f"Redundant (2x) PA:       {redundant_mean*1000:.2f} ms  (n={N_TRIALS})")
print(f"Overhead ratio:          {redundant_mean/single_mean:.2f}x")


Single PA extraction:    1.32 ms  (n=30)
Redundant (2x) PA:       1.42 ms  (n=30)
Overhead ratio:          1.08x


## TMR: compute overhead (local, Mock -- clean 3x-vs-1x comparison)

In [3]:
GEN_ALICE = RESULTS / "fabric_alice_sifted_bits_genonly.json"
GEN_BOB = RESULTS / "fabric_bob_sifted_bits_genonly.json"
if not (GEN_ALICE.exists() and GEN_BOB.exists()):
    raise FileNotFoundError("Adjust these paths to wherever your real key pair's sifted bits live.")

alice_key, _ = key_from_sifted_json(str(GEN_ALICE), "alice_bits")
bob_key, _ = key_from_sifted_json(str(GEN_BOB), "bob_bits")
qber = 0.0128  # use this key's real measured QBER

N_TRIALS = 10  # reconciliation is slower than PA extraction -- fewer trials is fine
single_times, tmr_times = [], []

for trial in range(N_TRIALS):
    seed = trial

    t0 = time.perf_counter()
    session = MockClassicalSession(correct_key=alice_key)
    recon = Reconciliation(algorithm=ORIGINAL, classical_session=session, noisy_key=bob_key,
                             estimated_bit_error_rate=qber, seed=seed, correct_key=alice_key)
    recon.reconcile()
    single_times.append(time.perf_counter() - t0)

    t0 = time.perf_counter()
    for rep in range(3):
        session = MockClassicalSession(correct_key=alice_key)
        recon = Reconciliation(algorithm=ORIGINAL, classical_session=session, noisy_key=bob_key,
                                 estimated_bit_error_rate=qber, seed=seed + rep * 1000, correct_key=alice_key)
        recon.reconcile()
    tmr_times.append(time.perf_counter() - t0)

single_mean, tmr_mean = np.mean(single_times), np.mean(tmr_times)
print(f"Single reconciliation:   {single_mean*1000:.1f} ms  (n={N_TRIALS})")
print(f"TMR (3x reconciliation): {tmr_mean*1000:.1f} ms  (n={N_TRIALS})")
print(f"Overhead ratio:          {tmr_mean/single_mean:.2f}x")


Single reconciliation:   22.5 ms  (n=10)
TMR (3x reconciliation): 60.5 ms  (n=10)
Overhead ratio:          2.69x


## TMR: real-channel overhead (FABRIC wall-clock, for context only)

This one is genuinely 3x by construction -- three sequential real-channel reconciliation
runs vs. one -- but per-run network latency varies, so report a range/CI rather than a
single clean multiplier if you cite this in the paper. Reuses your existing
`sdc_realchannel_reconciliation_*.csv`'s `elapsed_seconds` column rather than re-running
anything on FABRIC.

In [4]:
reconciliation_clean = pd.read_csv(str(RESULTS / "reconciliation_clean.csv"))
if "elapsed_seconds" in reconciliation_clean.columns:
    single_run_mean = reconciliation_clean["elapsed_seconds"].mean()
    single_run_std = reconciliation_clean["elapsed_seconds"].std()
    print(f"Single real-channel reconciliation run: {single_run_mean:.2f}s +/- {single_run_std:.2f}s")
    print(f"TMR (3 sequential real-channel runs), by construction: ~{3*single_run_mean:.2f}s "
          f"(propagated std: ~{np.sqrt(3)*single_run_std:.2f}s)")
else:
    print("elapsed_seconds column not found -- check reconciliation_clean.csv's columns")


Single real-channel reconciliation run: 3.91s +/- 1.23s
TMR (3 sequential real-channel runs), by construction: ~11.73s (propagated std: ~2.13s)
